In [1]:
import pandas as pd
import pickle
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib import pyplot as plt
from sklearn.preprocessing import RobustScaler, MinMaxScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score, accuracy_score, precision_score, f1_score, recall_score, confusion_matrix
from sklearn.metrics.pairwise import cosine_similarity

## Get songs in Spotify data format (e.g. danceability, energy, valence, ...)

In [2]:
#df = pd.read_csv("../data/raw/1200_song_mapped.csv",",")
df = pd.read_csv('../data/raw/full_survey_w_all_spotify_data.csv')
df.head()

,Unnamed: 0,index,Response_Id,Timestamp,Current_Mood,Recent_Songs,Recent_Movies,Music_Genre,Movie_Genre,Mood_to_Entrtnmnt,...,valence,tempo,type,id,uri,track_href,analysis_url,duration_ms,time_signature,popularity
0,0,0,0,9/28/2023 17:40:17,"Afraid, Excited","Pink - Lizzo, Cuff It - Beyonce, Bad Idea Righ...","Notes of Autumn, You Are So Not Invited To My ...","Pop, R&B, Punk","Action, Comedy, Romantic",3,...,0.803,114.995,audio_features,2YXGbxICUdOUJe9OPlicy1,spotify:track:2YXGbxICUdOUJe9OPlicy1,https://api.spotify.com/v1/tracks/2YXGbxICUdOU...,https://api.spotify.com/v1/audio-analysis/2YXG...,143833,4,76
1,1,1,0,9/28/2023 17:40:17,"Afraid, Excited","Pink - Lizzo, Cuff It - Beyonce, Bad Idea Righ...","Notes of Autumn, You Are So Not Invited To My ...","Pop, R&B, Punk","Action, Comedy, Romantic",3,...,0.642,115.042,audio_features,1xzi1Jcr7mEi9K2RfzLOqS,spotify:track:1xzi1Jcr7mEi9K2RfzLOqS,https://api.spotify.com/v1/tracks/1xzi1Jcr7mEi...,https://api.spotify.com/v1/audio-analysis/1xzi...,225389,4,86
2,2,2,0,9/28/2023 17:40:17,"Afraid, Excited","Pink - Lizzo, Cuff It - Beyonce, Bad Idea Righ...","Notes of Autumn, You Are So Not Invited To My ...","Pop, R&B, Punk","Action, Comedy, Romantic",3,...,0.748,129.979,audio_features,3IX0yuEVvDbnqUwMBB3ouC,spotify:track:3IX0yuEVvDbnqUwMBB3ouC,https://api.spotify.com/v1/tracks/3IX0yuEVvDbn...,https://api.spotify.com/v1/audio-analysis/3IX0...,184784,4,94
3,3,3,0,9/28/2023 17:40:17,"Afraid, Excited","Pink - Lizzo, Cuff It - Beyonce, Bad Idea Righ...","Notes of Autumn, You Are So Not Invited To My ...","Pop, R&B, Punk","Action, Comedy, Romantic",3,...,0.350,138.005,audio_features,1kuGVB7EU95pJObxwvfwKS,spotify:track:1kuGVB7EU95pJObxwvfwKS,https://api.spotify.com/v1/tracks/1kuGVB7EU95p...,https://api.spotify.com/v1/audio-analysis/1kuG...,219724,4,96
4,4,4,0,9/28/2023 17:40:17,"Afraid, Excited","Pink - Lizzo, Cuff It - Beyonce, Bad Idea Righ...","Notes of Autumn, You Are So Not Invited To My ...","Pop, R&B, Punk","Action, Comedy, Romantic",3,...,0.775,110.056,audio_features,1vYXt7VSjH9JIM5oRRo7vA,spotify:track:1vYXt7VSjH9JIM5oRRo7vA,https://api.spotify.com/v1/tracks/1vYXt7VSjH9J...,https://api.spotify.com/v1/audio-analysis/1vYX...,176579,4,95


In [3]:
# Drop unnecessary columns
df = df.drop(['Unnamed: 0', 'index','Timestamp'],axis=1)

In [4]:
df.columns

Index(['Response_Id', 'Current_Mood', 'Recent_Songs', 'Recent_Movies',
       'Music_Genre', 'Movie_Genre', 'Mood_to_Entrtnmnt', 'Music_to_Movie',
       'Artist', 'Track', 'audio_features', 'track_info', 'danceability',
       'energy', 'key', 'loudness', 'mode', 'speechiness', 'acousticness',
       'instrumentalness', 'liveness', 'valence', 'tempo', 'type', 'id', 'uri',
       'track_href', 'analysis_url', 'duration_ms', 'time_signature',
       'popularity'],
      dtype='object')

In [5]:
# josie_df = pd.read_csv('../data/raw/one_song.csv')
josie_df = pd.read_csv('../data/raw/josie_songs_v2.csv')

josie_df.head()

,Spotify ID,Artist IDs,Track Name,Album Name,Artist Name(s),Release Date,Duration (ms),Popularity,Added By,Added At,...,Key,Loudness,Mode,Speechiness,Acousticness,Instrumentalness,Liveness,Valence,Tempo,Time Signature
0,7njKgVl3AnLflRad38o7AG,4GvEc3ANtPPjt1ZJllr5Zl,Paradise,Paradise,Bazzi,2019-04-04,169038,0,spotify:user:jojorugjosie,2020-07-19T23:50:07Z,...,11,-6.273,0,0.0479,0.0828,0.00000,0.1130,0.591,122.061,4
1,2is19a6HY7ZdWP654IMhEE,37M5pPGs6V1fchFJSgCguX,Break My Heart Again,Break My Heart Again,FINNEAS,2018-02-09,238012,0,spotify:user:jojorugjosie,2020-07-19T23:51:35Z,...,4,-14.227,0,0.0395,0.9070,0.00000,0.0941,0.483,144.293,4
2,0gySkOOewOdZtHG8stPEB8,55fhWPvDiMpLnE4ZzNXZyW,Young & Sad,THE END OF EVERYTHING,Noah Cyrus,2020-05-15,190182,58,spotify:user:jojorugjosie,2020-07-19T23:51:39Z,...,7,-8.500,0,0.0380,0.5920,0.00000,0.1920,0.359,142.021,4
3,2wT9ZttmUbrxjNdhU9WvG1,"5qa31A9HySw3T7MKWI9bGg,3FLUBwpAnaIlIKeaBfsxFe",Bitter,Bitter,"FLETCHER,Kito",2020-05-13,194749,0,spotify:user:jojorugjosie,2020-07-19T23:51:43Z,...,6,-7.126,0,0.1020,0.3930,0.00116,0.0889,0.413,136.002,4
4,1OeEpViPwVi1FfdNkOoIci,"7wg1qvie3KqDNQbAkTdbX0,3ApUX1o6oSz321MMECyIYd,...",Little Things (with Quinn XCII & Chelsea Cutler),Little Things (with Quinn XCII & Chelsea Cutler),"Louis The Child,Quinn XCII,Chelsea Cutler",2020-04-03,197448,45,spotify:user:jojorugjosie,2020-07-19T23:52:28Z,...,6,-4.380,1,0.1220,0.1910,0.00000,0.0782,0.528,82.962,4


In [6]:
josie_df.columns = josie_df.columns.str.lower()

df = josie_df

In [7]:
# Raw data that will be fed into the robust scaler and then model
X = df[['danceability','acousticness','energy','instrumentalness','liveness','valence','loudness','speechiness','tempo']]
X

,danceability,acousticness,energy,instrumentalness,liveness,valence,loudness,speechiness,tempo
0,0.844,0.0828,0.644,0.000000,0.1130,0.591,-6.273,0.0479,122.061
1,0.402,0.9070,0.151,0.000000,0.0941,0.483,-14.227,0.0395,144.293
2,0.618,0.5920,0.428,0.000000,0.1920,0.359,-8.500,0.0380,142.021
3,0.724,0.3930,0.594,0.001160,0.0889,0.413,-7.126,0.1020,136.002
4,0.622,0.1910,0.636,0.000000,0.0782,0.528,-4.380,0.1220,82.962
...,...,...,...,...,...,...,...,...,...
66,0.590,0.2060,0.558,0.000134,0.5250,0.376,-7.055,0.1510,106.058
67,0.560,0.4790,0.418,0.000141,0.1090,0.334,-9.281,0.3960,175.996
68,0.559,0.0110,0.503,0.856000,0.4350,0.136,-10.660,0.0388,140.016
69,0.751,0.0485,0.377,0.000468,0.2910,0.140,-9.800,0.0538,112.478


## Preprocess songs' features for model

In [8]:
# Load robust scaler
loaded_rb = pickle.load(open("rb_scaler.pkl", "rb"))

/Users/jojorugjosie/opt/anaconda3/lib/python3.9/site-packages/sklearn/base.py:310: UserWarning: Trying to unpickle estimator RobustScaler from version 1.0.2 when using version 0.24.2. This might lead to breaking code or invalid results. Use at your own risk.
  warnings.warn(


In [9]:
# Transform the features
X_transformed = loaded_rb.transform(X)
X_transformed

/Users/jojorugjosie/opt/anaconda3/lib/python3.9/site-packages/sklearn/utils/validation.py:571: FutureWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  array.dtypes.apply(is_sparse).any()):


array([[ 1.33751306e+00, -1.44327007e-01,  9.11196911e-02,
        -6.53315110e-03,  1.09679188e-02,  5.96923077e-01,
         1.91002877e-01,  1.98501873e-01,  5.43667894e-02],
       [-5.09926855e-01,  7.79948975e-01, -6.70270270e-01,
        -6.53315110e-03, -1.96325747e-01,  3.31076923e-01,
        -5.98006150e-01, -1.16104869e-01,  7.17385145e-01],
       [ 3.92894462e-01,  4.26701057e-01, -2.42471042e-01,
        -6.53315110e-03,  8.77433507e-01,  2.58461538e-02,
        -2.99077472e-02, -1.72284644e-01,  6.49627962e-01],
       [ 8.35945664e-01,  2.03538086e-01,  1.38996139e-02,
        -5.12451629e-03, -2.53358925e-01,  1.58769231e-01,
         1.06388255e-01,  2.22471910e+00,  4.70125106e-01],
       [ 4.09613375e-01, -2.29891502e-02,  7.87644788e-02,
        -6.53315110e-03, -3.70715657e-01,  4.41846154e-01,
         3.78781867e-01,  2.97378277e+00, -1.11167112e+00],
       [ 1.79728318e-01,  7.03692282e-01, -4.91119691e-01,
        -6.53137816e-03, -2.52262133e-01, -7.121230

## Run model on songs and append back to dataframe

In [10]:
# Load model
filename = "music_to_mood_model.pickle"
loaded_model = pickle.load(open(filename, "rb"))

/Users/jojorugjosie/opt/anaconda3/lib/python3.9/site-packages/sklearn/base.py:310: UserWarning: Trying to unpickle estimator DecisionTreeClassifier from version 1.0.2 when using version 0.24.2. This might lead to breaking code or invalid results. Use at your own risk.
  warnings.warn(
/Users/jojorugjosie/opt/anaconda3/lib/python3.9/site-packages/sklearn/base.py:310: UserWarning: Trying to unpickle estimator RandomForestClassifier from version 1.0.2 when using version 0.24.2. This might lead to breaking code or invalid results. Use at your own risk.
  warnings.warn(


In [11]:
# Get predictions on data
y_pred = loaded_model.predict(X_transformed)

In [12]:
# Mapping emotions
emotions_mapping = {0: 'sad', 1: 'happy', 2:'energetic', 3:'calm'}

In [13]:
# Concatenate mood predictions to main dataframe
df['mood'] = y_pred
df['mood'] = df['mood'].map(emotions_mapping)
df

,spotify id,artist ids,track name,album name,artist name(s),release date,duration (ms),popularity,added by,added at,...,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time signature,mood
0,7njKgVl3AnLflRad38o7AG,4GvEc3ANtPPjt1ZJllr5Zl,Paradise,Paradise,Bazzi,2019-04-04,169038,0,spotify:user:jojorugjosie,2020-07-19T23:50:07Z,...,-6.273,0,0.0479,0.0828,0.000000,0.1130,0.591,122.061,4,happy
1,2is19a6HY7ZdWP654IMhEE,37M5pPGs6V1fchFJSgCguX,Break My Heart Again,Break My Heart Again,FINNEAS,2018-02-09,238012,0,spotify:user:jojorugjosie,2020-07-19T23:51:35Z,...,-14.227,0,0.0395,0.9070,0.000000,0.0941,0.483,144.293,4,sad
2,0gySkOOewOdZtHG8stPEB8,55fhWPvDiMpLnE4ZzNXZyW,Young & Sad,THE END OF EVERYTHING,Noah Cyrus,2020-05-15,190182,58,spotify:user:jojorugjosie,2020-07-19T23:51:39Z,...,-8.500,0,0.0380,0.5920,0.000000,0.1920,0.359,142.021,4,sad
3,2wT9ZttmUbrxjNdhU9WvG1,"5qa31A9HySw3T7MKWI9bGg,3FLUBwpAnaIlIKeaBfsxFe",Bitter,Bitter,"FLETCHER,Kito",2020-05-13,194749,0,spotify:user:jojorugjosie,2020-07-19T23:51:43Z,...,-7.126,0,0.1020,0.3930,0.001160,0.0889,0.413,136.002,4,happy
4,1OeEpViPwVi1FfdNkOoIci,"7wg1qvie3KqDNQbAkTdbX0,3ApUX1o6oSz321MMECyIYd,...",Little Things (with Quinn XCII & Chelsea Cutler),Little Things (with Quinn XCII & Chelsea Cutler),"Louis The Child,Quinn XCII,Chelsea Cutler",2020-04-03,197448,45,spotify:user:jojorugjosie,2020-07-19T23:52:28Z,...,-4.380,1,0.1220,0.1910,0.000000,0.0782,0.528,82.962,4,happy
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
66,0yUH4cVZ9usfGwru3RpeCd,5L67d6oZKOOmXcBcMMJdZZ,Hands,Hands,Point Point,2017-04-28,217358,42,spotify:user:jojorugjosie,2023-04-21T01:33:36Z,...,-7.055,0,0.1510,0.2060,0.000134,0.5250,0.376,106.058,4,happy
67,2x4Wu8WR2X47rMBgWVWBnU,"6nS5roXSAGhTGr34W6n7Et,163tK9Wjr9P9DmM0AVK7lm,...",Magnets - SG Lewis Remix,Magnets (The Remixes),"Disclosure,Lorde,SG Lewis",2015-11-16,286363,42,spotify:user:jojorugjosie,2023-04-21T01:39:30Z,...,-9.281,0,0.3960,0.4790,0.000141,0.1090,0.334,175.996,4,sad
68,4SLw3mYAeDLzg0gLp8pi2x,7KGI0OwY8iAL5rI5p47YyK,Get Like It,Tora,Tora,2013-10-07,129427,45,spotify:user:jojorugjosie,2023-04-21T01:45:57Z,...,-10.660,0,0.0388,0.0110,0.856000,0.4350,0.136,140.016,4,sad
69,0BAQjCC9gfqOebxpmonHKz,"62vbsDRAq0qHdezaCOzB0T,5EBlHXi71tDXnFtroEh7Rg",Everytime,RIPPLS,"Medasin,Naomi Wild",2020-03-06,170884,53,spotify:user:jojorugjosie,2023-04-21T02:01:22Z,...,-9.800,1,0.0538,0.0485,0.000468,0.2910,0.140,112.478,3,happy


In [14]:
# Concatenate mood PROBABILITY predictions to main dataframe
df[['sad','happy','energetic','calm']] = loaded_model.predict_proba(X_transformed)
df

,spotify id,artist ids,track name,album name,artist name(s),release date,duration (ms),popularity,added by,added at,...,instrumentalness,liveness,valence,tempo,time signature,mood,sad,happy,energetic,calm
0,7njKgVl3AnLflRad38o7AG,4GvEc3ANtPPjt1ZJllr5Zl,Paradise,Paradise,Bazzi,2019-04-04,169038,0,spotify:user:jojorugjosie,2020-07-19T23:50:07Z,...,0.000000,0.1130,0.591,122.061,4,happy,0.02,0.85,0.13,0.00
1,2is19a6HY7ZdWP654IMhEE,37M5pPGs6V1fchFJSgCguX,Break My Heart Again,Break My Heart Again,FINNEAS,2018-02-09,238012,0,spotify:user:jojorugjosie,2020-07-19T23:51:35Z,...,0.000000,0.0941,0.483,144.293,4,sad,0.92,0.05,0.00,0.03
2,0gySkOOewOdZtHG8stPEB8,55fhWPvDiMpLnE4ZzNXZyW,Young & Sad,THE END OF EVERYTHING,Noah Cyrus,2020-05-15,190182,58,spotify:user:jojorugjosie,2020-07-19T23:51:39Z,...,0.000000,0.1920,0.359,142.021,4,sad,0.93,0.05,0.01,0.01
3,2wT9ZttmUbrxjNdhU9WvG1,"5qa31A9HySw3T7MKWI9bGg,3FLUBwpAnaIlIKeaBfsxFe",Bitter,Bitter,"FLETCHER,Kito",2020-05-13,194749,0,spotify:user:jojorugjosie,2020-07-19T23:51:43Z,...,0.001160,0.0889,0.413,136.002,4,happy,0.31,0.54,0.14,0.01
4,1OeEpViPwVi1FfdNkOoIci,"7wg1qvie3KqDNQbAkTdbX0,3ApUX1o6oSz321MMECyIYd,...",Little Things (with Quinn XCII & Chelsea Cutler),Little Things (with Quinn XCII & Chelsea Cutler),"Louis The Child,Quinn XCII,Chelsea Cutler",2020-04-03,197448,45,spotify:user:jojorugjosie,2020-07-19T23:52:28Z,...,0.000000,0.0782,0.528,82.962,4,happy,0.11,0.72,0.17,0.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
66,0yUH4cVZ9usfGwru3RpeCd,5L67d6oZKOOmXcBcMMJdZZ,Hands,Hands,Point Point,2017-04-28,217358,42,spotify:user:jojorugjosie,2023-04-21T01:33:36Z,...,0.000134,0.5250,0.376,106.058,4,happy,0.25,0.66,0.06,0.03
67,2x4Wu8WR2X47rMBgWVWBnU,"6nS5roXSAGhTGr34W6n7Et,163tK9Wjr9P9DmM0AVK7lm,...",Magnets - SG Lewis Remix,Magnets (The Remixes),"Disclosure,Lorde,SG Lewis",2015-11-16,286363,42,spotify:user:jojorugjosie,2023-04-21T01:39:30Z,...,0.000141,0.1090,0.334,175.996,4,sad,0.68,0.24,0.00,0.08
68,4SLw3mYAeDLzg0gLp8pi2x,7KGI0OwY8iAL5rI5p47YyK,Get Like It,Tora,Tora,2013-10-07,129427,45,spotify:user:jojorugjosie,2023-04-21T01:45:57Z,...,0.856000,0.4350,0.136,140.016,4,sad,0.46,0.24,0.09,0.21
69,0BAQjCC9gfqOebxpmonHKz,"62vbsDRAq0qHdezaCOzB0T,5EBlHXi71tDXnFtroEh7Rg",Everytime,RIPPLS,"Medasin,Naomi Wild",2020-03-06,170884,53,spotify:user:jojorugjosie,2023-04-21T02:01:22Z,...,0.000468,0.2910,0.140,112.478,3,happy,0.40,0.50,0.07,0.03


In [15]:
from scipy.stats import boxcox

# Extract var3 and var4 columns
var3 = df['energetic']
var4 = df['calm']

# Apply Box-Cox transformation
transformed_var3, lambda_var3 = boxcox(var3 + 1)  # Adding 1 to handle zero values
transformed_var4, lambda_var4 = boxcox(var4 + 1)

# Scale the transformed variables to the range [0, 1]
scaler = MinMaxScaler()
scaled_var3 = scaler.fit_transform(transformed_var3.reshape(-1, 1))
scaled_var4 = scaler.fit_transform(transformed_var4.reshape(-1, 1))

# Update the DataFrame with scaled variables
df['energetic_scaled'] = scaled_var3
df['calm_scaled'] = scaled_var4

In [16]:
# Assuming df is your DataFrame
df['mood_scaled'] = df[['happy', 'sad', 'energetic_scaled', 'calm_scaled']].idxmax(axis=1)
# df['mood_scaled'] = df[['happy', 'sad', 'energetic', 'calm']].idxmax(axis=1)

# Assuming df is your DataFrame
mood_scaled_counts = df['mood_scaled'].value_counts()

# # Display the count of each unique value
# print(mood_scaled_counts)
# # Display the DataFrame with the new 'mood_scaled' column
# df

# CLUSTERING

In [17]:
## CLUSTERING VERSION 1: 5 CLUSTERS ##

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# Assuming df is your DataFrame with columns 'happy', 'sad', 'energetic', 'calm'
# You may need to adjust the column names or use the actual DataFrame structure

if df.shape[0] == 1:
    df['cluster_v1'] = 0
else:
    # Select the columns for clustering
    columns_for_clustering = ['danceability','acousticness','energy','instrumentalness','liveness','valence','loudness','speechiness','tempo']

    # Extract the feature matrix
    X = df[columns_for_clustering]

    # Standardize the feature matrix
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    # Specify the number of clusters
    num_clusters = 5

    # Apply K-Means clustering
    kmeans = KMeans(n_clusters=num_clusters, random_state=42)
    df['cluster_v1'] = kmeans.fit_predict(X_scaled)

# Display the resulting DataFrame with cluster assignments
df

/Users/jojorugjosie/opt/anaconda3/lib/python3.9/site-packages/sklearn/utils/validation.py:571: FutureWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  array.dtypes.apply(is_sparse).any()):
/Users/jojorugjosie/opt/anaconda3/lib/python3.9/site-packages/sklearn/utils/validation.py:571: FutureWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  array.dtypes.apply(is_sparse).any()):


,spotify id,artist ids,track name,album name,artist name(s),release date,duration (ms),popularity,added by,added at,...,time signature,mood,sad,happy,energetic,calm,energetic_scaled,calm_scaled,mood_scaled,cluster_v1
0,7njKgVl3AnLflRad38o7AG,4GvEc3ANtPPjt1ZJllr5Zl,Paradise,Paradise,Bazzi,2019-04-04,169038,0,spotify:user:jojorugjosie,2020-07-19T23:50:07Z,...,4,happy,0.02,0.85,0.13,0.00,0.476948,0.000000,happy,1
1,2is19a6HY7ZdWP654IMhEE,37M5pPGs6V1fchFJSgCguX,Break My Heart Again,Break My Heart Again,FINNEAS,2018-02-09,238012,0,spotify:user:jojorugjosie,2020-07-19T23:51:35Z,...,4,sad,0.92,0.05,0.00,0.03,0.000000,0.624660,sad,2
2,0gySkOOewOdZtHG8stPEB8,55fhWPvDiMpLnE4ZzNXZyW,Young & Sad,THE END OF EVERYTHING,Noah Cyrus,2020-05-15,190182,58,spotify:user:jojorugjosie,2020-07-19T23:51:39Z,...,4,sad,0.93,0.05,0.01,0.01,0.048264,0.280759,sad,2
3,2wT9ZttmUbrxjNdhU9WvG1,"5qa31A9HySw3T7MKWI9bGg,3FLUBwpAnaIlIKeaBfsxFe",Bitter,Bitter,"FLETCHER,Kito",2020-05-13,194749,0,spotify:user:jojorugjosie,2020-07-19T23:51:43Z,...,4,happy,0.31,0.54,0.14,0.01,0.503043,0.280759,happy,1
4,1OeEpViPwVi1FfdNkOoIci,"7wg1qvie3KqDNQbAkTdbX0,3ApUX1o6oSz321MMECyIYd,...",Little Things (with Quinn XCII & Chelsea Cutler),Little Things (with Quinn XCII & Chelsea Cutler),"Louis The Child,Quinn XCII,Chelsea Cutler",2020-04-03,197448,45,spotify:user:jojorugjosie,2020-07-19T23:52:28Z,...,4,happy,0.11,0.72,0.17,0.00,0.574767,0.000000,happy,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
66,0yUH4cVZ9usfGwru3RpeCd,5L67d6oZKOOmXcBcMMJdZZ,Hands,Hands,Point Point,2017-04-28,217358,42,spotify:user:jojorugjosie,2023-04-21T01:33:36Z,...,4,happy,0.25,0.66,0.06,0.03,0.256836,0.624660,happy,0
67,2x4Wu8WR2X47rMBgWVWBnU,"6nS5roXSAGhTGr34W6n7Et,163tK9Wjr9P9DmM0AVK7lm,...",Magnets - SG Lewis Remix,Magnets (The Remixes),"Disclosure,Lorde,SG Lewis",2015-11-16,286363,42,spotify:user:jojorugjosie,2023-04-21T01:39:30Z,...,4,sad,0.68,0.24,0.00,0.08,0.000000,0.923100,calm_scaled,4
68,4SLw3mYAeDLzg0gLp8pi2x,7KGI0OwY8iAL5rI5p47YyK,Get Like It,Tora,Tora,2013-10-07,129427,45,spotify:user:jojorugjosie,2023-04-21T01:45:57Z,...,4,sad,0.46,0.24,0.09,0.21,0.359947,1.000000,calm_scaled,3
69,0BAQjCC9gfqOebxpmonHKz,"62vbsDRAq0qHdezaCOzB0T,5EBlHXi71tDXnFtroEh7Rg",Everytime,RIPPLS,"Medasin,Naomi Wild",2020-03-06,170884,53,spotify:user:jojorugjosie,2023-04-21T02:01:22Z,...,3,happy,0.40,0.50,0.07,0.03,0.292837,0.624660,calm_scaled,2


In [18]:
## CLUSTERING VERSION 2: DYNAMIC CLUSTERING ##

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
import numpy as np

# Assuming df is your DataFrame with columns 'happy', 'sad', 'energetic_scaled', 'calm_scaled'
# You may need to adjust the column names or use the actual DataFrame structure

if df.shape[0] == 1:
    df['cluster_v2'] = 0
    cluster_counts = df['cluster_v2'].value_counts().sort_index()
    cluster_song_counts = cluster_counts.to_list()
else:
    # Select the columns for clustering
    columns_for_clustering = ['danceability','acousticness','energy','instrumentalness','liveness',
                              'valence','loudness','speechiness','tempo']

    # Extract the feature matrix
    X = df[columns_for_clustering]

    # Standardize the feature matrix
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    # Initialize variables
    best_num_clusters = 1  # Start with a minimum number of clusters
    best_silhouette_score = -1  # Initialize with a low value

    # Loop through different cluster numbers
    for num_clusters in range(1, min(6, len(X) - 1)):  # Try up to 5 clusters or less based on # of data points
        if num_clusters == 1:
            df['cluster_v2'] = 0
        else:
            kmeans = KMeans(n_clusters=num_clusters, random_state=42)
            df['cluster_v2'] = kmeans.fit_predict(X_scaled)

            # Calculate silhouette score
            silhouette_avg = silhouette_score(X_scaled, df['cluster_v2'])

            # Check if the current number of clusters improves silhouette score
            if silhouette_avg > best_silhouette_score:
                best_silhouette_score = silhouette_avg
                best_num_clusters = num_clusters

    # Apply K-Means clustering with the best number of clusters
    kmeans = KMeans(n_clusters=best_num_clusters, random_state=42)
    df['cluster_v2'] = kmeans.fit_predict(X_scaled)

    # Count the number of songs in each cluster
    cluster_counts = df['cluster_v2'].value_counts().sort_index()
    cluster_song_counts = cluster_counts.to_list()

# Display the resulting DataFrame with cluster assignments
print(cluster_counts)
print(cluster_song_counts)

cluster_v2
0     5
1    35
2    21
3     2
4     8
Name: count, dtype: int64
[5, 35, 21, 2, 8]


/Users/jojorugjosie/opt/anaconda3/lib/python3.9/site-packages/sklearn/utils/validation.py:571: FutureWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  array.dtypes.apply(is_sparse).any()):
/Users/jojorugjosie/opt/anaconda3/lib/python3.9/site-packages/sklearn/utils/validation.py:571: FutureWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  array.dtypes.apply(is_sparse).any()):


In [19]:
# Assuming you have a DataFrame df with a 'cluster_v2' column
# and columns_for_clustering as the features used for clustering

# Select the columns for clustering
columns_for_clustering = ['danceability','acousticness','energy','instrumentalness','liveness',
                          'valence','loudness','speechiness','tempo']

# Group by the 'cluster_v2' column and calculate descriptive statistics
cluster_stats = df.groupby('cluster_v2')[columns_for_clustering].describe()

# Display the range (min and max) for each feature in each cluster
print("Cluster Descriptive Statistics:")
for cluster in range(len(cluster_stats)):
    print(f"\nCluster {cluster}:")
    print(cluster_stats.loc[cluster, [('danceability', 'min'), ('danceability', 'max'),
                                      ('acousticness', 'min'), ('acousticness', 'max'),
                                      ('energy', 'min'), ('energy', 'max'),
                                      ('instrumentalness', 'min'), ('instrumentalness', 'max'),
                                      ('liveness', 'min'), ('liveness', 'max'),
                                      ('valence', 'min'), ('valence', 'max'),
                                      ('loudness', 'min'), ('loudness', 'max'),
                                      ('speechiness', 'min'), ('speechiness', 'max'),
                                      ('tempo', 'min'), ('tempo', 'max')]])

# Alternatively, you can use describe() for a more comprehensive overview
# print(cluster_stats)


Cluster Descriptive Statistics:

Cluster 0:
danceability      min      0.554000
                  max      0.773000
acousticness      min      0.016900
                  max      0.230000
energy            min      0.498000
                  max      0.747000
instrumentalness  min      0.000000
                  max      0.000134
liveness          min      0.439000
                  max      0.965000
valence           min      0.165000
                  max      0.413000
loudness          min     -8.866000
                  max     -4.425000
speechiness       min      0.043600
                  max      0.151000
tempo             min     79.635000
                  max    130.124000
Name: 0, dtype: float64

Cluster 1:
danceability      min      0.39200
                  max      0.84400
acousticness      min      0.00656
                  max      0.66500
energy            min      0.42500
                  max      0.94900
instrumentalness  min      0.00000
                  max      

In [20]:
# from sklearn.cluster import KMeans
# from sklearn.preprocessing import StandardScaler
# from sklearn.metrics import silhouette_score
# import numpy as np

# # Assuming df is your DataFrame with columns 'danceability','acousticness','energy',...

# if df.shape[0] == 1:
#     df['cluster_v2'] = 0
#     cluster_counts = df['cluster_v2'].value_counts().sort_index()
#     cluster_song_counts = cluster_counts.to_list()
# else:
#     # Select the columns for clustering
#     columns_for_clustering = ['danceability','acousticness','energy','instrumentalness','liveness',
#                               'valence','loudness','speechiness','tempo']

#     # Extract the feature matrix
#     X = df[columns_for_clustering]

#     # Standardize the feature matrix
#     scaler = StandardScaler()
#     X_scaled = scaler.fit_transform(X)

#     # Initialize variables
#     best_num_clusters = 1  # Start with a minimum number of clusters
#     best_silhouette_score = -1  # Initialize with a low value

#     # Set your custom threshold for the range of values for a cluster
#     custom_threshold = 0.7

#     # Loop through different cluster numbers
#     for num_clusters in range(1, min(6, len(X) - 1)):  # Try up to 5 clusters or less based on # of data points
#         if num_clusters == 1:
#             df['cluster_v2'] = 0
#         else:
#             kmeans = KMeans(n_clusters=num_clusters, random_state=42)
#             df['cluster_v2'] = kmeans.fit_predict(X_scaled)

#             # Calculate silhouette score
#             silhouette_avg = silhouette_score(X_scaled, df['cluster_v2'])

#             # Check if the current number of clusters improves silhouette score and meets the custom threshold
#             if silhouette_avg > best_silhouette_score and silhouette_avg > custom_threshold:
#                 best_silhouette_score = silhouette_avg
#                 best_num_clusters = num_clusters

#     # Apply K-Means clustering with the best number of clusters
#     kmeans = KMeans(n_clusters=best_num_clusters, random_state=42)
#     df['cluster_v2'] = kmeans.fit_predict(X_scaled)

#     # Count the number of songs in each cluster
#     cluster_counts = df['cluster_v2'].value_counts().sort_index()
#     cluster_song_counts = cluster_counts.to_list()

# # Display the resulting DataFrame with cluster assignments
# print(cluster_counts)
# print(cluster_song_counts)


In [21]:
df

,spotify id,artist ids,track name,album name,artist name(s),release date,duration (ms),popularity,added by,added at,...,mood,sad,happy,energetic,calm,energetic_scaled,calm_scaled,mood_scaled,cluster_v1,cluster_v2
0,7njKgVl3AnLflRad38o7AG,4GvEc3ANtPPjt1ZJllr5Zl,Paradise,Paradise,Bazzi,2019-04-04,169038,0,spotify:user:jojorugjosie,2020-07-19T23:50:07Z,...,happy,0.02,0.85,0.13,0.00,0.476948,0.000000,happy,1,1
1,2is19a6HY7ZdWP654IMhEE,37M5pPGs6V1fchFJSgCguX,Break My Heart Again,Break My Heart Again,FINNEAS,2018-02-09,238012,0,spotify:user:jojorugjosie,2020-07-19T23:51:35Z,...,sad,0.92,0.05,0.00,0.03,0.000000,0.624660,sad,2,2
2,0gySkOOewOdZtHG8stPEB8,55fhWPvDiMpLnE4ZzNXZyW,Young & Sad,THE END OF EVERYTHING,Noah Cyrus,2020-05-15,190182,58,spotify:user:jojorugjosie,2020-07-19T23:51:39Z,...,sad,0.93,0.05,0.01,0.01,0.048264,0.280759,sad,2,2
3,2wT9ZttmUbrxjNdhU9WvG1,"5qa31A9HySw3T7MKWI9bGg,3FLUBwpAnaIlIKeaBfsxFe",Bitter,Bitter,"FLETCHER,Kito",2020-05-13,194749,0,spotify:user:jojorugjosie,2020-07-19T23:51:43Z,...,happy,0.31,0.54,0.14,0.01,0.503043,0.280759,happy,1,1
4,1OeEpViPwVi1FfdNkOoIci,"7wg1qvie3KqDNQbAkTdbX0,3ApUX1o6oSz321MMECyIYd,...",Little Things (with Quinn XCII & Chelsea Cutler),Little Things (with Quinn XCII & Chelsea Cutler),"Louis The Child,Quinn XCII,Chelsea Cutler",2020-04-03,197448,45,spotify:user:jojorugjosie,2020-07-19T23:52:28Z,...,happy,0.11,0.72,0.17,0.00,0.574767,0.000000,happy,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
66,0yUH4cVZ9usfGwru3RpeCd,5L67d6oZKOOmXcBcMMJdZZ,Hands,Hands,Point Point,2017-04-28,217358,42,spotify:user:jojorugjosie,2023-04-21T01:33:36Z,...,happy,0.25,0.66,0.06,0.03,0.256836,0.624660,happy,0,0
67,2x4Wu8WR2X47rMBgWVWBnU,"6nS5roXSAGhTGr34W6n7Et,163tK9Wjr9P9DmM0AVK7lm,...",Magnets - SG Lewis Remix,Magnets (The Remixes),"Disclosure,Lorde,SG Lewis",2015-11-16,286363,42,spotify:user:jojorugjosie,2023-04-21T01:39:30Z,...,sad,0.68,0.24,0.00,0.08,0.000000,0.923100,calm_scaled,4,4
68,4SLw3mYAeDLzg0gLp8pi2x,7KGI0OwY8iAL5rI5p47YyK,Get Like It,Tora,Tora,2013-10-07,129427,45,spotify:user:jojorugjosie,2023-04-21T01:45:57Z,...,sad,0.46,0.24,0.09,0.21,0.359947,1.000000,calm_scaled,3,3
69,0BAQjCC9gfqOebxpmonHKz,"62vbsDRAq0qHdezaCOzB0T,5EBlHXi71tDXnFtroEh7Rg",Everytime,RIPPLS,"Medasin,Naomi Wild",2020-03-06,170884,53,spotify:user:jojorugjosie,2023-04-21T02:01:22Z,...,happy,0.40,0.50,0.07,0.03,0.292837,0.624660,calm_scaled,2,2


## Average the outputted happy/sad/energetic/calm vectors

In [22]:
# Get data for one user
# df_one_user = df[df['Response_Id']==0]
# df_one_user

In [23]:
user_mood_vector = df[['happy', 'sad', 'energetic_scaled', 'calm_scaled']].mean()
# user_mood_vector = df[['happy', 'sad', 'energetic', 'calm']].mean()
user_mood_vector

happy               0.485563
sad                 0.354648
energetic_scaled    0.419231
calm_scaled         0.335197
dtype: float64

In [24]:
# Reshape mood vector for cosine similarity 
user_mood_reshaped = user_mood_vector.values.reshape(1, -1)

## Load movie data and perform cosine similarity to get top 5 movie recommendations

#### NOTE: Make sure you unzip file below before reading

In [25]:
movie_df = pd.read_csv('../data/raw/wiki_movies_mood_label_by_gpt-3.5-turbo_final.csv').loc[:,'year':]
movie_df

,year,href,title,plot,premise,synopsis,directed_by,screenplay_by,based_on,starring,...,roi,roi_binary,roi_multi,rt_binary,rt_multi,happy,sad,energetic,calm,error_labeling
0,1973,40_Carats_(film),40 Carats (film),"Ann Stanley, who sells real estate in New York...",NaN,NaN,Directed byMilton Katselas,Screenplay byLeonard Gershe,Based on\nQuarante caratsby Pierre BarilletJea...,Starring\nLiv Ullmann\nEdward Albert\nGene Kel...,...,NaN,0,0,0,0,0.8,0.3,0.6,0.7,NaN
1,1973,Ace_Eli_and_Rodger_of_the_Skies,Ace Eli and Rodger of the Skies,"In the early 1920s, Eli (Cliff Robertson) is a...",NaN,NaN,Directed byJohn Erman,Screenplay byClaudia Salter,NaN,Starring\nCliff Robertson\nEric Shea\nPamela F...,...,NaN,0,0,0,0,0.7,0.6,0.8,0.5,NaN
2,1973,The_Affair_(1973_film),The Affair (1973 film),Courtney Patterson is a beautiful 32 year old ...,NaN,NaN,Directed byGilbert Cates,NaN,NaN,StarringNatalie WoodRobert WagnerBruce Davison...,...,NaN,0,0,0,0,0.5,0.8,0.4,0.7,NaN
3,1973,The_All-American_Boy_(film),The All-American Boy (film),"Vic Bealer, a young boxer from a small town in...",NaN,NaN,Directed byCharles Eastman,NaN,NaN,StarringJon VoightE. J. Peaker,...,NaN,0,0,0,0,0.2,0.6,0.8,0.4,NaN
4,1973,American_Graffiti,American Graffiti,On their last evening of summer vacation in 19...,NaN,NaN,Directed byGeorge Lucas,NaN,NaN,Starring\nRichard Dreyfuss\nRon Howard\nPaul L...,...,180.18018,1,2,1,2,0.5,0.7,0.7,0.5,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11933,2023,Rebel_Moon,Rebel Moon,NaN,In a universe controlled by the corrupt govern...,NaN,Directed byZack Snyder,Screenplay by\nZack Snyder\nKurt Johnstad\nSha...,NaN,Starring\nSofia Boutella\nCharlie Hunnam\nMich...,...,NaN,0,0,0,0,0.3,0.4,0.8,0.5,NaN
11934,2023,The_Iron_Claw_(film),The Iron Claw (film),NaN,NaN,The film is centered around the Von Erich fami...,Directed bySean Durkin,NaN,NaN,Starring\nZac Efron\nJeremy Allen White\nHarri...,...,NaN,0,0,0,0,0.3,0.7,0.8,0.2,NaN
11935,2023,The_Color_Purple_(2023_film),The Color Purple (2023 film),NaN,A story of the life-long struggles of an Afric...,NaN,Directed byBlitz Bazawule,Screenplay byMarcus Gardley,Based on\nThe Color Purpleby Alice Walker\nThe...,Starring\nTaraji P. Henson\nDanielle Brooks\nC...,...,NaN,0,0,0,0,0.1,0.9,0.2,0.8,NaN
11936,2023,The_Boys_in_the_Boat_(film),The Boys in the Boat (film),The non-fiction novel describes the University...,NaN,NaN,Directed byGeorge Clooney,Screenplay byMark L. Smith,Based onThe Boys in the Boatby Daniel James Brown,Starring\nCallum Turner\nJoel Edgerton\n,...,NaN,0,0,0,0,0.2,0.7,0.8,0.4,NaN


In [26]:
movie_df.columns

Index(['year', 'href', 'title', 'plot', 'premise', 'synopsis', 'directed_by',
       'screenplay_by', 'based_on', 'starring', 'box_office', 'written_by',
       'budget', 'omdb_response', 'omdb_title', 'genres', 'omdb_plot',
       'release_date', 'omdb_director', 'omdb_writer', 'omdb_actors',
       'imdb_score', 'imdb_votes', 'rotten_tomatoes_score', 'metacritic_score',
       'directed_by_clean', 'screenplay_by_clean', 'written_by_clean',
       'starring_clean', 'based_on_binary', 'based_on_binary_characters',
       'based_on_binary_comic_books', 'box_office_clean', 'budget_clean',
       'chosen_plot_feature', 'writers_to_use', 'actors_to_use',
       'plot_parenthetical', 'num_parentheticals',
       'plot_parenthetical_fuzzy_scores', 'new_plot',
       'all_writers_first_movie', 'any_writers_first_movie', 'roi',
       'roi_binary', 'roi_multi', 'rt_binary', 'rt_multi', 'happy', 'sad',
       'energetic', 'calm', 'error_labeling'],
      dtype='object')

In [27]:
filt_movie_df = movie_df.dropna(subset=['rotten_tomatoes_score']).loc[movie_df['imdb_score'] >= 5.0]
filt_movie_df = filt_movie_df[filt_movie_df['year'] >= 1995]
filt_movie_df.head()

,year,href,title,plot,premise,synopsis,directed_by,screenplay_by,based_on,starring,...,roi,roi_binary,roi_multi,rt_binary,rt_multi,happy,sad,energetic,calm,error_labeling
4476,1995,Die_Hard_with_a_Vengeance,Die Hard with a Vengeance,The Bonwit Teller department store in New York...,NaN,NaN,Directed byJohn McTiernan,Screenplay byJonathan Hensleigh,Based on\nSimon Saysby Jonathan Hensleigh\nCha...,Starring\nBruce Willis\nJeremy Irons\nSamuel L...,...,4.067778,1,2,0,0,0.6,0.3,0.9,0.2,NaN
4477,1995,Toy_Story,Toy Story,A group of living toys are preparing to move i...,NaN,NaN,Directed byJohn Lasseter,Screenplay by\nJoss Whedon\nAndrew Stanton\nJo...,NaN,Starring\nTom Hanks\nTim Allen\nLaurie Metcalf...,...,13.146667,1,2,1,0,0.8,0.4,0.9,0.3,NaN
4478,1995,Apollo_13_(film),Apollo 13 (film),"On July 20, 1969, astronaut Jim Lovell hosts a...",NaN,NaN,Directed byRon Howard,Screenplay byWilliam Broyles Jr. Al Reinert,Based onLost Moonby Jim LovellJeffrey Kluger,Starring\nTom Hanks\nKevin Bacon\nBill Paxton\...,...,6.830769,1,2,1,0,0.7,0.4,0.8,0.6,NaN
4479,1995,GoldenEye,GoldenEye,"In 1986, MI6 agents James Bond and Alec Trevel...",NaN,NaN,Directed byMartin Campbell,Screenplay by\nJeffrey Caine\nBruce Feirstein\n,Based onJames Bondby Ian Fleming,Starring\nPierce Brosnan\nSean Bean\nIzabella ...,...,5.940000,1,2,1,0,0.4,0.3,0.8,0.5,NaN
4480,1995,Pocahontas_(1995_film),Pocahontas (1995 film),"In 1607, the Susan Constant travels from Londo...",NaN,NaN,Directed by\nMike Gabriel\nEric Goldberg\n,NaN,Based onPocahontas,Starring\nJoe Baker\nChristian Bale\nIrene Bed...,...,6.292727,1,2,0,0,0.8,0.4,0.6,0.9,NaN


In [28]:
# Load mood vectors for movies
# movie_mood_vectors = movie_df[['happy','sad','energetic','calm']].values
movie_mood_vectors = filt_movie_df[['happy','sad','energetic','calm']].values
movie_mood_vectors

array([[0.6, 0.3, 0.9, 0.2],
       [0.8, 0.4, 0.9, 0.3],
       [0.7, 0.4, 0.8, 0.6],
       ...,
       [0.2, 0.7, 0.5, 0.6],
       [0.7, 0.3, 0.8, 0.5],
       [0.7, 0.3, 0.9, 0.4]])

# SIMPLE COSINE SIMILARITY

In [29]:
## VERSION 1: WITHOUT CLUSTERING ##

# Perform cosine similarity to get top 5 similar movies based on mood vector
from sklearn.metrics.pairwise import cosine_similarity

similarities = cosine_similarity(user_mood_reshaped, movie_mood_vectors)
sim_scores = list(enumerate(similarities[0]))
sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

# Get top movie titles based on top similarity scores
top_movies = sim_scores[1:6]
top_movie_indices = [i[0] for i in top_movies]
recommended_movies = filt_movie_df.iloc[top_movie_indices]
recommended_movies

,year,href,title,plot,premise,synopsis,directed_by,screenplay_by,based_on,starring,...,roi,roi_binary,roi_multi,rt_binary,rt_multi,happy,sad,energetic,calm,error_labeling
10979,2020,Safety_(2020_film),Safety (2020 film),"Set in South Carolina in the 2000s, Ray McElra...",NaN,NaN,Directed byReginald Hudlin,NaN,NaN,Starring\nJay Reeves\nThaddeus J. Mixson\nCori...,...,NaN,0,0,1,0,0.8,0.6,0.7,0.6,NaN
11803,2023,Haunted_Mansion_(2023_film),Haunted Mansion (2023 film),"During a New Year's party, Ben Matthias, an as...",NaN,NaN,Directed byJustin Simien,Screenplay byKatie Dippold,Based onThe Haunted Mansionby Walt Disney,Starring\nLaKeith Stanfield\nTiffany Haddish\n...,...,0.761533,0,0,0,0,0.8,0.6,0.7,0.6,NaN
5093,1996,A_Time_to_Kill_(1996_film),A Time to Kill (1996 film),"In Canton, Mississippi, ten-year-old African A...",NaN,NaN,Directed byJoel Schumacher,Screenplay byAkiva Goldsman,Based onA Time to Killby John Grisham,Starring\nSandra Bullock\nSamuel L. Jackson\nM...,...,3.800000,1,2,1,0,0.9,0.7,0.8,0.6,NaN
5379,1997,Pooh%27s_Grand_Adventure:_The_Search_for_Chris...,Pooh's Grand Adventure: The Search for Christo...,"Once upon the last day of a golden summer, Chr...",NaN,NaN,Directed byKarl Geurs,NaN,Based onCharacters createdby A. A. Milne,Starring\nJim Cummings\nJohn Fiedler\nKen Sans...,...,NaN,0,0,0,0,0.9,0.7,0.8,0.6,NaN
5904,1999,Random_Hearts,Random Hearts,"Sergeant William ""Dutch"" Van Den Broeck is a p...",NaN,NaN,Directed bySydney Pollack,Screenplay byKurt Luedtke\n,Based onRandom Heartsby Warren Adler,Starring\nHarrison Ford\nKristin Scott Thomas\...,...,1.165625,1,1,0,0,0.9,0.7,0.8,0.6,NaN


In [30]:
## VERSION 2: WITH CLUSTERING ##

result_df = pd.DataFrame(columns=['year', 'href', 'title', 'plot', 'premise', 'synopsis', 'directed_by',
       'screenplay_by', 'based_on', 'starring', 'box_office', 'written_by',
       'budget', 'omdb_response', 'omdb_title', 'genres', 'omdb_plot',
       'release_date', 'omdb_director', 'omdb_writer', 'omdb_actors',
       'imdb_score', 'imdb_votes', 'rotten_tomatoes_score', 'metacritic_score',
       'directed_by_clean', 'screenplay_by_clean', 'written_by_clean',
       'starring_clean', 'based_on_binary', 'based_on_binary_characters',
       'based_on_binary_comic_books', 'box_office_clean', 'budget_clean',
       'chosen_plot_feature', 'writers_to_use', 'actors_to_use',
       'plot_parenthetical', 'num_parentheticals',
       'plot_parenthetical_fuzzy_scores', 'new_plot',
       'all_writers_first_movie', 'any_writers_first_movie', 'roi',
       'roi_binary', 'roi_multi', 'rt_binary', 'rt_multi', 'happy', 'sad',
       'energetic', 'calm', 'error_labeling'])

# Iterate over each cluster
if df.shape[0] == 1:
    similarities = cosine_similarity(user_mood_reshaped, movie_mood_vectors)
    sim_scores = list(enumerate(similarities[0]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    # Get top movie titles based on top similarity scores
    top_movies = sim_scores[1:6]
    top_movie_indices = [i[0] for i in top_movies]
    result_df = filt_movie_df.iloc[top_movie_indices]
else:
    for user_cluster in range(5):
        # Step 2: Calculate the average mood vector for each cluster
        cluster_avg_mood = df.groupby('cluster_v1')[['happy', 'sad', 'energetic_scaled', 'calm_scaled']].mean()
#         cluster_avg_mood = df.groupby('cluster_v1')[['happy', 'sad', 'energetic', 'calm']].mean()

        # Reshape the user's cluster average mood vector
        user_mood_reshaped = cluster_avg_mood.loc[user_cluster].values.reshape(1, -1)

        # Step 3: Use cosine similarity to find the top movies similar to the average mood vectors of the clusters
        similarities = cosine_similarity(user_mood_reshaped, movie_mood_vectors)
        sim_scores = list(enumerate(similarities[0]))
        sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

        # Get top movie titles based on top similarity scores
        top_movies = sim_scores[1][0]
    #     top_movie_indices = [i[0] for i in top_movies]
        recommended_movies = filt_movie_df.iloc[top_movies]
        result_df.loc[len(result_df)] = recommended_movies

result_df

/var/folders/c9/59t_gt3d35749qwchmywrq9c0000gn/T/ipykernel_61391/1494161103.py:46: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  result_df.loc[len(result_df)] = recommended_movies


,year,href,title,plot,premise,synopsis,directed_by,screenplay_by,based_on,starring,...,roi,roi_binary,roi_multi,rt_binary,rt_multi,happy,sad,energetic,calm,error_labeling
0,1996,When_We_Were_Kings,When We Were Kings,NaN,NaN,NaN,Directed byLeon Gast,NaN,NaN,Starring\nMuhammad Ali\nGeorge Foreman\nJames ...,...,NaN,0,0,1,0,0.8,0.2,0.9,0.4,NaN
1,2011,Rio_(2011_film),Rio (2011 film),"In Moose Lake, Minnesota, a crate containing a...",NaN,NaN,Directed byCarlos Saldanha,Screenplay by\nDon Rhymer\nJoshua Sternin\nJen...,NaN,Starring\nJesse Eisenberg\nAnne Hathaway\nwill...,...,5.384444,1,2,1,0,1.0,0.2,0.8,0.3,NaN
2,2012,The_Fitzgerald_Family_Christmas,The Fitzgerald Family Christmas,NaN,The seven adult siblings of the Fitzgerald fam...,NaN,Directed byEdward Burns,NaN,NaN,StarringKerry BishéEdward BurnsHeather BurnsMa...,...,NaN,0,0,1,0,0.4,0.8,0.2,0.6,NaN
3,2003,Girl_with_a_Pearl_Earring_(film),Girl with a Pearl Earring (film),Griet is a shy girl living in the Dutch Republ...,NaN,NaN,Directed byPeter Webber,Screenplay byOlivia Hetreed,Based onGirl with a Pearl Earringby Tracy Chev...,Starring\nColin Firth\nScarlett Johansson\nTom...,...,NaN,0,0,1,0,0.1,0.7,0.2,0.9,NaN
4,1995,The_American_President,The American President,Popular Democratic President Andrew Shepherd p...,NaN,NaN,Directed byRob Reiner,NaN,NaN,Starring\nMichael Douglas\nAnnette Bening\nMar...,...,1.740323,1,1,1,0,0.8,0.4,0.9,0.7,NaN


In [31]:
## VERSION 3 PART 1: WITH DYNAMIC CLUSTERING ##

cluster_movie_counts = {}
for i in range(len(cluster_song_counts)):
    cluster_movie_counts[i] = int(np.round(5 * (cluster_song_counts[i] / sum(cluster_song_counts))))

# If the amount of movies recommended is less than 5, add a movie to the cluster with the most songs
while sum(cluster_movie_counts.values()) < 5:
    max_cluster = max(cluster_movie_counts, key=cluster_movie_counts.get)
    cluster_movie_counts[max_cluster] += 1

# If the amount of movies recommended is more than 5, remove a movie from the cluster with the most songs
while sum(cluster_movie_counts.values()) > 5:
    max_cluster = max(cluster_movie_counts, key=cluster_movie_counts.get)
    cluster_movie_counts[max_cluster] -= 1
    
print(cluster_movie_counts)

{0: 0, 1: 3, 2: 1, 3: 0, 4: 1}


In [32]:
## VERSION 3 PART 2: WITH DYNAMIC CLUSTERING ##

result_df = pd.DataFrame(columns=['year', 'href', 'title', 'plot', 'premise', 'synopsis', 'directed_by',
       'screenplay_by', 'based_on', 'starring', 'box_office', 'written_by',
       'budget', 'omdb_response', 'omdb_title', 'genres', 'omdb_plot',
       'release_date', 'omdb_director', 'omdb_writer', 'omdb_actors',
       'imdb_score', 'imdb_votes', 'rotten_tomatoes_score', 'metacritic_score',
       'directed_by_clean', 'screenplay_by_clean', 'written_by_clean',
       'starring_clean', 'based_on_binary', 'based_on_binary_characters',
       'based_on_binary_comic_books', 'box_office_clean', 'budget_clean',
       'chosen_plot_feature', 'writers_to_use', 'actors_to_use',
       'plot_parenthetical', 'num_parentheticals',
       'plot_parenthetical_fuzzy_scores', 'new_plot',
       'all_writers_first_movie', 'any_writers_first_movie', 'roi',
       'roi_binary', 'roi_multi', 'rt_binary', 'rt_multi', 'happy', 'sad',
       'energetic', 'calm', 'error_labeling'])

for user_cluster, num_movies in cluster_movie_counts.items():
    # Calculate the average mood vector for each cluster
    cluster_avg_mood = df.groupby('cluster_v2')[['happy', 'sad', 'energetic_scaled', 'calm_scaled']].mean()
#     cluster_avg_mood = df.groupby('cluster_v2')[['happy', 'sad', 'energetic', 'calm']].mean()

    # Reshape the user's cluster average mood vector
    user_mood_reshaped = cluster_avg_mood.loc[user_cluster].values.reshape(1, -1)

    # Use cosine similarity to find the top movies similar to the average mood vectors of the clusters
    similarities = cosine_similarity(user_mood_reshaped, movie_mood_vectors)
    sim_scores = list(enumerate(similarities[0]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    # Get top movie titles based on top similarity scores
    top_movies = [i[0] for i in sim_scores[1:num_movies+1]]
    recommended_movies = filt_movie_df.iloc[top_movies]
#     df.loc[len(df.index)] =
    result_df = pd.concat([result_df, recommended_movies], ignore_index=True)

result_df.reset_index(drop=True, inplace=True)

result_df

/var/folders/c9/59t_gt3d35749qwchmywrq9c0000gn/T/ipykernel_61391/814379453.py:35: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  result_df = pd.concat([result_df, recommended_movies], ignore_index=True)


,year,href,title,plot,premise,synopsis,directed_by,screenplay_by,based_on,starring,...,roi,roi_binary,roi_multi,rt_binary,rt_multi,happy,sad,energetic,calm,error_labeling
0,2011,Rio_(2011_film),Rio (2011 film),"In Moose Lake, Minnesota, a crate containing a...",NaN,NaN,Directed byCarlos Saldanha,Screenplay by\nDon Rhymer\nJoshua Sternin\nJen...,NaN,Starring\nJesse Eisenberg\nAnne Hathaway\nwill...,...,5.384444,1,2,1,0,1.0,0.2,0.8,0.3,NaN
1,1995,Jumanji_(film),Jumanji (film),"In 1969, Alan Parrish lives with his parents, ...",NaN,NaN,Directed byJoe Johnston,Screenplay by\nJonathan Hensleigh\nGreg Taylor...,Based onJumanjiby Chris Van Allsburg,Starring\nRobin Williams\nKirsten Dunst\nDavid...,...,4.043077,1,2,0,0,0.8,0.2,0.7,0.3,NaN
2,1995,The_Amazing_Panda_Adventure,The Amazing Panda Adventure,Panda rescuer Michael Tyler sends his son Ryan...,NaN,NaN,Directed byChristopher Cain,Screenplay byJeff RothbergLaurice Elehwany,NaN,Starring\nStephen Lang\nYi Ding\nRyan Slater\n,...,NaN,0,0,0,0,0.8,0.2,0.7,0.3,NaN
3,2012,The_Fitzgerald_Family_Christmas,The Fitzgerald Family Christmas,NaN,The seven adult siblings of the Fitzgerald fam...,NaN,Directed byEdward Burns,NaN,NaN,StarringKerry BishéEdward BurnsHeather BurnsMa...,...,NaN,0,0,1,0,0.4,0.8,0.2,0.6,NaN
4,1995,The_American_President,The American President,Popular Democratic President Andrew Shepherd p...,NaN,NaN,Directed byRob Reiner,NaN,NaN,Starring\nMichael Douglas\nAnnette Bening\nMar...,...,1.740323,1,1,1,0,0.8,0.4,0.9,0.7,NaN
